# ethical wall rag v2

Runtime: A100 + high RAM. No secrets needed, all models and data are public on Hugging Face.
Outputs are copied to Drive under `ethical-wall-rag/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip install -q "git+https://github.com/arihaaant/ethical-wall-rag.git@v2"

In [ ]:
import json, shutil, time
from google.colab import drive

drive.mount('/content/drive')
stamp = time.strftime('%Y%m%d-%H%M')
out = f'/content/results/{stamp}'
drive_out = f'/content/drive/MyDrive/ethical-wall-rag/{stamp}'
print(drive_out)

## full run: retrieval comparison, red team, then Qwen2.5-7B answers

In [ ]:
from ethicalwall import run_v2

t0 = time.time()
run_v2.main(['--out', out])
print(f'{(time.time() - t0) / 60:.1f} min')
shutil.copytree(out, drive_out, dirs_exist_ok=True)

## tables

In [ ]:
import pandas as pd

s = json.load(open(f'{out}/summary.json'))
pct = lambda d: None if d is None else round(100 * d['rate'], 1)

pd.DataFrame({name: {'hit@5 all': pct(v['all']), 'clauses': pct(v['clauses']),
                     'metadata': pct(v['metadata']), 'top1': pct(v['top1'])}
              for name, v in s['retrieval'].items()}).T

In [ ]:
pd.DataFrame({mode: {'leak %': pct(v['leak']), 'leak % when code in context': pct(v['leak_when_exposed']),
                     'forbidden code in context %': pct(v['forbidden_code_in_context']),
                     'correct on allowed %': pct(v['correct_on_allowed'])}
              for mode, v in s['access_control'].items()}).T

In [ ]:
pd.DataFrame({t: {mode: pct(s['access_control'][mode]['leak_by_template'][t]) for mode in s['access_control']}
              for t in s['access_control']['prompting']['leak_by_template']}).T

In [ ]:
print('injection followed:', {k: pct(v) for k, v in s['injection'].items()})
print('answer quality:', {k: (round(v, 3) if isinstance(v, float) else pct(v)) for k, v in s['answer_quality'].items()})
print('red team (bge retrieval):', {k: (pct(v) if isinstance(v, dict) else round(v, 2)) for k, v in s['redteam_bge'].items()})

In [ ]:
# a few answers to read by hand
ac = pd.read_csv(f'{out}/access_control.csv')
leaks = ac[(ac['mode'] == 'prompting') & ac.leak]
for r in leaks.head(5).itertuples():
    print(r.user, '|', r.q, '\n ->', r.answer[:300], '\n')

In [ ]:
inj = pd.read_csv(f'{out}/injection.csv')
for r in inj[inj.followed].head(3).itertuples():
    print('hardened' if r.harden else 'plain', '|', r.q, '\n ->', r.answer[:200], '\n')